In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, Any, Tuple, Optional, Iterable, List
import logging
import numpy as np
import xarray as xr

# -------------------------------------------------------------
# Helpers communs
# -------------------------------------------------------------

def _match_levels(avail_vals, requested, tol: Optional[float]) -> List[float]:
    """Fait correspondre une liste de niveaux Ã  partir d'une tolÃ©rance optionnelle.
    - Si tol est None, exige l'Ã©galitÃ© exacte.
    - Sinon, prend la valeur la plus proche si |diff| <= tol, et dÃ©doublonne.
    """
    if not requested:
        return []
    if tol is None:
        avail = set(avail_vals)
        return [lv for lv in requested if lv in avail]

    vals = np.asarray(avail_vals, dtype=float)
    out = []
    for lv in requested:
        lvf = float(lv)
        j = int(np.argmin(np.abs(vals - lvf)))
        if abs(float(vals[j]) - lvf) <= tol:
            out.append(float(vals[j]))
    seen = set(); uniq = []
    for v in out:
        if v not in seen:
            seen.add(v); uniq.append(v)
    return uniq


def _normalize_nc_requests(raw: Optional[Dict[str, Dict[str, Any]]]) -> Dict[str, Dict[str, Any]]:
    """Normalise un bloc YAML pour NetCDF.
    ClÃ©s supportÃ©es: "level", "level_w", "surface".
    Chaque entrÃ©e devient un dict avec: variables, all_levels, level_indices, levels.
    """
    if not raw:
        return {}
    out: Dict[str, Dict[str, Any]] = {}
    for group, spec in raw.items():
        spec = spec or {}
        out[group] = {
            "variables": list(spec.get("variables", [])),
            "all_levels": bool(spec.get("all_levels", False) or str(spec.get("level_selection", "")).lower().strip() == "all"),
            "level_indices": list(spec.get("level_indices", [])) if spec.get("level_selection", "").lower() != "values" else [],
            "levels": list(spec.get("level_values", spec.get("levels", []))) if str(spec.get("level_selection", "")).lower() in {"values", ""} else [],
        }
    return out


def _build_group_ds(
    ds: xr.Dataset,
    req: Dict[str, Any],
    group_key: str,
    concat_dimension: str,
    float_tol: Optional[float],
    logger: Optional[logging.Logger],
) -> Optional[xr.Dataset]:
    """Construit un xr.Dataset pour un groupe donnÃ© Ã  partir d'un ds global.
    group_key in {"level", "level_w", "surface"}.
    """
    vars_req = req.get("variables", [])
    all_levels = bool(req.get("all_levels", False))
    level_indices = req.get("level_indices")
    levels = req.get("levels")

    if group_key == "surface":
        keep = [v for v in vars_req if v in ds.data_vars and ("level" not in ds[v].dims) and ("level_w" not in ds[v].dims)]
        if not keep:
            return None
        var_map = {}
        for v in keep:
            da = ds[v]
            if concat_dimension in ds.dims and concat_dimension not in da.dims:
                da = da.expand_dims({concat_dimension: ds[concat_dimension]})
            var_map[v] = da
        return xr.Dataset(var_map) if var_map else None

    # vertical groups
    vert_dim = group_key
    keep = [v for v in vars_req if v in ds.data_vars and (vert_dim in ds[v].dims)]
    if not keep:
        return None

    var_map = {}
    for v in keep:
        da = ds[v]
        if not all_levels and vert_dim in da.dims:
            if level_indices is not None:
                n = da.sizes.get(vert_dim, 0)
                idx_list = [int(i) for i in level_indices]
                valid = [i for i in idx_list if -n <= i < n]
                if len(valid) < len(idx_list):
                    bad = [i for i in idx_list if i not in valid]
                    msg = f"[concat_nc2ds_two_requests_by_vert_coord] indices hors plage {bad} pour taille {n} (var={v}, dim={vert_dim})"
                    (logger.warning(msg) if logger else print(msg))
                if valid:
                    da = da.isel({vert_dim: valid})
                else:
                    continue
            elif levels is not None:
                if vert_dim not in da.coords:
                    msg = f"[concat_nc2ds_two_requests_by_vert_coord] coord '{vert_dim}' absente pour var={v}"
                    (logger.warning(msg) if logger else print(msg))
                    continue
                avail = da[vert_dim].values.tolist()
                matched = _match_levels(avail, levels, tol=float_tol)
                missing = [lv for lv in levels if lv not in matched]
                if missing:
                    msg = f"[concat_nc2ds_two_requests_by_vert_coord] niveaux non appariÃ©s {missing} (var={v}, dim={vert_dim})"
                    (logger.warning(msg) if logger else print(msg))
                if matched:
                    da = da.sel({vert_dim: matched})
                else:
                    continue
        if concat_dimension in ds.dims and concat_dimension not in da.dims:
            da = da.expand_dims({concat_dimension: ds[concat_dimension]})
        var_map[v] = da

    return xr.Dataset(var_map) if var_map else None


# -------------------------------------------------------------
# Ouverture NetCDF minimaliste, rÃ©utilisable
# -------------------------------------------------------------

def concat_nc2ds(
    files_list: Iterable[str | Path],
    variables: Iterable[str],
    concat_dimension: str | None = None,
    parallel: bool = False,
    logger: Optional[logging.Logger] = None,
    strict: bool = False,
) -> xr.Dataset:
    files = [Path(f) for f in files_list if Path(f).exists()]
    if not files:
        raise FileNotFoundError("Aucun fichier NetCDF valide trouvÃ©.")
    files = sorted(files)

    # Liste des variables du premier fichier (pour drop_variables)
    with xr.open_dataset(files[0], decode_cf=False, engine="h5netcdf") as t0:
        all_vars = list(t0.data_vars)
    keep_vars = list(dict.fromkeys([v for v in variables if v in all_vars]))
    drop_vars = [v for v in all_vars if v not in keep_vars]

    if len(files) == 1:
        if logger: logger.info("Mode mono-fichier: xr.open_dataset")
        ds = xr.open_dataset(
            files[0],
            engine="h5netcdf",
            drop_variables=drop_vars or None,
        )
    else:
        if logger:
            logger.info(
                "Mode multi-fichiers: xr.open_mfdataset (combine=%s, concat_dim=%s)",
                "nested" if concat_dimension else "by_coords",
                concat_dimension,
            )
        ds = xr.open_mfdataset(
            files,
            engine="h5netcdf",
            drop_variables=drop_vars or None,
            combine="nested" if concat_dimension else "by_coords",
            concat_dim=concat_dimension,
            parallel=parallel,
        )

    if logger:
        present = ", ".join(sorted(ds.data_vars))
        logger.info("Variables dans le dataset: %s", present)
    return ds


# -------------------------------------------------------------
# API principale: 2 requÃªtes (user + tracker)
# -------------------------------------------------------------

def concat_nc2ds_two_requests_by_vert_coord(
    files: Iterable[str | Path],
    user_requested_variables_yaml: Dict[str, Dict[str, Any]],
    tracker_requested_variables_by_method_yaml: Dict[str, Dict[str, Any]] | None,
    method: Optional[str] = None,
    *,
    concat_dimension: str = "time",
    parallel: bool = True,
    strict: bool = False,
    logger: Optional[logging.Logger] = None,
    keep_geovars: bool = True,
    geovar_candidates: tuple[str, ...] = ("latitude", "longitude"),
    float_tol: Optional[float] = None,
) -> Tuple[Dict[str, xr.Dataset], Dict[str, Dict[str, xr.Dataset]]]:
    """
    Ouvre les NetCDF une seule fois puis applique deux requÃªtes indÃ©pendantes.
    Retourne (by_group_user, by_group_tracker), avec:
      - by_group_user: {group -> xr.Dataset}
      - by_group_tracker: {method -> {group -> xr.Dataset}}

    Groupes supportÃ©s: "level", "level_w", "surface".
    - SÃ©lection verticale par indices ou valeurs. `float_tol=None` impose l'Ã©galitÃ© exacte.
    - Les variables rÃ©ellement lues sont l'union des variables demandÃ©es par la requÃªte user
      et toutes les mÃ©thodes tracker sÃ©lectionnÃ©es, plus les variables gÃ©ographiques si prÃ©sentes.
    - `method=None` traite toutes les mÃ©thodes prÃ©sentes dans `tracker_requested_variables_by_method_yaml`.
    """
    logger = logger or logging.getLogger("frameit")

    # Normalisation requÃªte user
    user_req = _normalize_nc_requests(user_requested_variables_yaml)

    # SÃ©lection et normalisation des mÃ©thodes tracker
    tracker_yaml = tracker_requested_variables_by_method_yaml or {}
    if method is None:
        methods = list(tracker_yaml.keys())
    else:
        methods = [m for m in tracker_yaml.keys() if m == method]
        if not methods:
            logger.warning("MÃ©thode tracker '%s' absente. Aucun groupe tracker ne sera produit.", method)
    tracker_req_by_method: Dict[str, Dict[str, Any]] = {
        m: _normalize_nc_requests(tracker_yaml.get(m, {})) for m in methods
    }

    # Union minimale des variables Ã  lire
    requested_vars_all: set[str] = set()
    for _, spec in user_req.items():
        requested_vars_all.update(spec.get("variables", []))
    for m, blk in tracker_req_by_method.items():
        for _, spec in blk.items():
            requested_vars_all.update(spec.get("variables", []))

    # PrÃ©paration des fichiers
    files = [Path(f) for f in files if Path(f).exists()]
    if not files:
        raise FileNotFoundError("Aucun fichier NetCDF valide trouvÃ©.")
    files = sorted(files)

    # Ajout automatique de gÃ©ovariables
    auto_geo = []
    if keep_geovars:
        with xr.open_dataset(files[0], decode_cf=False, engine="h5netcdf") as t0:
            present = set(t0.data_vars) | set(t0.coords)
        auto_geo = [name for name in geovar_candidates if name in present]
        if auto_geo:
            logger.info("Ajout auto des variables gÃ©o: %s", ", ".join(auto_geo))
        requested_vars_all.update(auto_geo)

    # Lecture unique
    ds = concat_nc2ds(
        files_list=files,
        variables=sorted(requested_vars_all),
        concat_dimension=concat_dimension,
        parallel=parallel,
        logger=logger,
        strict=strict,
    )

    # Placer les gÃ©ovars en coords si besoin
    if auto_geo:
        keep_as_coords = [v for v in auto_geo if v in ds]
        if keep_as_coords:
            ds = ds.set_coords(keep_as_coords)

    # Construction des groupes
    out_user: Dict[str, xr.Dataset] = {}
    out_trk: Dict[str, Dict[str, xr.Dataset]] = {}

    for g in ("level", "level_w", "surface"):
        if g in user_req:
            ds_u = _build_group_ds(ds, user_req[g], g, concat_dimension, float_tol, logger)
            if ds_u is not None:
                out_user[g] = ds_u

    for m, blk in tracker_req_by_method.items():
        grp_map: Dict[str, xr.Dataset] = {}
        for g in ("level", "level_w", "surface"):
            if g in blk:
                ds_t = _build_group_ds(ds, blk[g], g, concat_dimension, float_tol, logger)
                if ds_t is not None:
                    grp_map[g] = ds_t
        if grp_map:
            out_trk[m] = grp_map

    return out_user, out_trk



In [ ]:
user_yaml = {
    "level":     {"variables": ["UT","VT","THT","REHU"], "level_indices": [0, 5, 10]},
    "level_w":   {"variables": ["WT"], "level_indices": [0, 5, 10]},
    "surface":   {"variables": ["MSLP","UM10","VM10"]},
}
trk_yaml = {
    "wind_pressure": {
        "level": {"variables": ["UT"], "level_indices": [2,10,15]},
        "surface": {"variables": ["UM10", "VM10", "MSLP"]},
        
    },
    "other_method": {
        "level": {"variables": ["VT"], "level_indices": [2,5,7]},
        "surface": {"variables": ["UM10", "LE", "MSLP"]},
        
    }
}
files = sorted(Path("/cnrm/mlac/NO_SAVE/souffletc/DONEE_TEST_FRAMEIT/CHIDO").glob("*.nc"))

out_user, out_trk = concat_nc2ds_two_requests_by_vert_coord(
    files, user_yaml, trk_yaml, method=None, logger=None, float_tol=None
)

In [ ]:
# out_trk.keys()
out_trk['wind_pressure']['level']


In [ ]:
by_user, by_trk = concat_nc2ds_two_requests_by_vert_coord(
    files,
    user_requested_variables_yaml=user_yaml,
    tracker_requested_variables_by_method_yaml=trk_yaml,
    method="wind_pressure",
    concat_dimension="time",
    float_tol=None,   # ou 0.5 si niveaux flottants
    logger=logging.getLogger("frameit"),
)